## Applying Average CounterFactual Difference with Shapley Values to Gridworld 


applying the novel XRL method ACD-SPV to a girdworld environment 

ronan kenny 

In [ ]:
from src.ACDEvaluator import ACDEvaluator
from src.FeatureMasker import FeatureMasker
from src.CoalitionPolicyRegistry import CoalitionPolicyRegistry
from src.ShapleyEngine import ShapleyEngine
import numpy as np

# Define Gridworld Parameters 
BOARD_ROWS = 6
BOARD_COLS = 6
WIN_STATE = (5, 5)
LOSE_STATE = (1, 4)
WALLS = [(2, 1), (2, 2),(3,1),(4,1)]  # Define walls as a list of tuples
START = (0, 0)
PENALTY = -0.7
STEP_PENALTY = -0.02
#DETERMINISTIC = False
DETERMINISTIC = True


# class to run a grid world environment 
class GridWorld1Env:


    # setup gridworld environment 
    def __init__(self):

        self.rows = BOARD_ROWS
        self.cols = BOARD_COLS

        self.actions = [0, 1, 2, 3]  # 0:Right, 1:Left, 2:Up, 3:Down <- discrete action set 
        self.action_names = {0: "Right", 1: "Left", 2: "Up", 3: "Down"}
        self.deltas = {0: (0, 1), 1: (0, -1), 2: (-1, 0), 3: (1, 0)}

    def get_features(self, state):
        near_lose = int(
            state != LOSE_STATE
            and abs(state[0] - LOSE_STATE[0]) + abs(state[1] - LOSE_STATE[1]) <= 1
        )
        return (*state, near_lose)

    def get_non_terminal_states(self):
        terminal_states = {WIN_STATE, LOSE_STATE}
        walls = set(WALLS)
        return [
            (r, c)
            for r in range(self.rows)
            for c in range(self.cols)
            if (r, c) not in terminal_states and (r, c) not in walls
        ]

# train pi policy 
    def step(self, state, action):
        if state in (WIN_STATE, LOSE_STATE):
            return state, 0.0, True

        dx, dy = self.deltas[action]
        nx = int(np.clip(state[0] + dx, 0, self.rows - 1))
        ny = int(np.clip(state[1] + dy, 0, self.cols - 1))
        proposed_state = (nx,ny)

        next_state = state if proposed_state in WALLS else proposed_state
        reward = STEP_PENALTY
        done = False

        if next_state == WIN_STATE:
            reward = 13
            done = True
        elif next_state == LOSE_STATE:
            reward = PENALTY
            done = True

        return next_state, reward, done


if __name__ == "__main__":
    np.random.seed(42)
    env = GridWorld1Env()

    # Wire up the 4 modules

    # Features: row, column, and whether the agent is adjacent to the lose state.
    masker = FeatureMasker(num_features=3, feature_extractor=env.get_features)

    registry = CoalitionPolicyRegistry(env, masker)

    # (Optional) If you already trained your full model, plug it in here:
    # registry.register_pretrained_full_model(my_policy, my_V, my_Q)

    
    # train all induced policies pi^C (skips any already registered) for each subset of features C
    registry.build_all(episodes=5000, gamma=0.95)

    # initialize ACD Evaluator & Shapley Engine
    # can choose which modes -(try mode="V" or mode="Q") based on state or action value function 
    acd_eval = ACDEvaluator(env, registry, mode="V")
    shapley = ShapleyEngine(masker)

    # Compare explanations away from and adjacent to the lose state.
    for label, s in [("Far", (1, 2)), ("Near", (1, 3))]:
        full_features = masker.project(s, masker.all_features)
        opt_a = registry.registry[masker.all_features]["policy"][full_features]
        scores = shapley.explain(s, acd_eval)
        print(
            f"{label} {s} | near_lose={full_features[2]} | "
            f"Best Action: {env.action_names[opt_a]:5s} | "
            f"ACD-SPV [row, col, near_lose]: {np.round(scores, 3)}"
        )

    

In [ ]:
# Trace the learned full-feature policy, recording the explanation at each state.
trajectory = []
state = START
visited_states = {state}
for step_index in range(50):
    full_features = masker.project(state, masker.all_features)
    action = registry.registry[masker.all_features]["policy"][full_features]
    scores = shapley.explain(state, acd_eval)
    next_state, reward, done = env.step(state, action)
    trajectory.append((step_index, state, full_features[2], action, reward, scores))
    state = next_state
    if done or state in visited_states:
        break
    visited_states.add(state)

print("\nGreedy policy trajectory from START:")
print("step  state    near_lose action  reward   row_phi  col_phi  near_phi")
for step_index, state, near_lose, action, reward, scores in trajectory:
    print(
        f"{step_index:>4}  {str(state):<8} {near_lose:>9} "
        f"{env.action_names[action]:<6} {reward:>7.2f} "
        f"{scores[0]:>8.3f} {scores[1]:>8.3f} {scores[2]:>9.3f}"
    )

path_states = {record[1] for record in trajectory}
feature_labels = ("row", "col", "near_lose")
print("\nACD-SPV grid by feature (visited cells are marked with *):")
for feature_index, feature_name in enumerate(feature_labels):
    print(f"\n{feature_name}:")
    print("       " + "".join(f"{col:^10}" for col in range(env.cols)))
    for row in range(env.rows):
        cells = []
        for col in range(env.cols):
            cell = (row, col)
            if cell in WALLS:
                label = "WALL"
            elif cell == WIN_STATE:
                label = "GOAL"
            elif cell == LOSE_STATE:
                label = "LOSE"
            else:
                score = shapley.explain(cell, acd_eval)[feature_index]
                marker = "*" if cell in path_states else " "
                label = f"{marker}{score:+.3f}"
            cells.append(f"{label:^10}")
        print(f"{row:>3}    " + "".join(cells))